# LangChain Complete Guide
## RAG | Agents | LangGraph | LangSmith

This notebook covers:
1. RAG Without LangChain (manual implementation)
2. RAG With LangChain (using LangChain components)
3. LangGraph Agent with Tools, Guardrails, and Orchestrator
4. LangSmith Tracing

Model used throughout: `gpt-4o-mini`

---
## SECTION 0 - Installation and Setup

In [ ]:
import subprocess, sys

packages = [
    "openai", "langchain", "langchain-openai", "langchain-community",
    "langchain-core", "langgraph", "langsmith",
    "faiss-cpu", "tiktoken", "numpy", "requests", "python-dotenv", "chromadb"
]

print("📦 Installing dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + packages, check=True)
print("✅ All dependencies installed successfully")

📦 Installing dependencies...
✅ All dependencies installed successfully



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

# ── API Keys ──────────────────────────────────────────────────────────────────
# Fill these in if not using a .env file
os.environ["OPENAI_API_KEY"]      = "sk-proj---_JkUyKXx2owDlixPWjuWFoNETNKToFO96ltaWLtL37XDDYevMA"


openai_key    = os.environ.get("OPENAI_API_KEY", "")


print("[SETUP] Environment variables configured.")
print(f"[SETUP] OpenAI key   : {'✅ set' if openai_key.startswith('sk-') else '❌ missing'}")


[SETUP] Environment variables configured.
[SETUP] OpenAI key   : ✅ set
[SETUP] LangSmith key: ✅ set


In [ ]:
from openai import OpenAI

client = OpenAI()   # reads OPENAI_API_KEY from env
print("✅ OpenAI client initialized")

✅ OpenAI client initialized


---
## SECTION 3 - LangGraph Agent

Stateful agent with:
- **Tools**: `get_weather`, `calculator`, `get_definition`
- **Guardrail**: keyword-based input filter
- **Graph**: `llm ↔ tools` loop with conditional routing

In [ ]:
# ── Tool Definitions ──────────────────────────────────────────────────────────
import math
import requests
from langchain_core.tools import tool


@tool
def get_weather(city: str) -> str:
    """Get current weather for a city (temperature °C, condition, humidity)."""
    print(f"[TOOL: get_weather] city='{city}'")
    api_key = os.environ.get("OPENWEATHER_API_KEY", "")
    if api_key:
        url = (
            f"http://api.openweathermap.org/data/2.5/weather"
            f"?q={city}&appid={api_key}&units=metric"
        )
        try:
            resp = requests.get(url, timeout=5)
            if resp.status_code == 200:
                d = resp.json()
                result = (
                    f"Weather in {city}: {d['main']['temp']}C, "
                    f"{d['weather'][0]['description']}, "
                    f"humidity {d['main']['humidity']}%"
                )
                print(f"[TOOL: get_weather] {result}")
                return result
        except Exception as e:
            print(f"[TOOL: get_weather] API error: {e} — falling back to mock")

    mock = {
        "london":    "Weather in London: 12C, overcast clouds, humidity 78%",
        "new york":  "Weather in New York: 18C, clear sky, humidity 55%",
        "tokyo":     "Weather in Tokyo: 22C, light rain, humidity 82%",
        "paris":     "Weather in Paris: 15C, partly cloudy, humidity 65%",
        "hyderabad": "Weather in Hyderabad: 35C, sunny, humidity 40%",
    }
    result = mock.get(city.lower(), f"Weather in {city}: 20C, clear sky, humidity 60% (mock)")
    print(f"[TOOL: get_weather] {result}")
    return result


@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression. Examples: '2+2', 'sqrt(16)', '100*1.08'."""
    print(f"[TOOL: calculator] expr='{expression}'")
    try:
        safe_ns = {k: v for k, v in math.__dict__.items() if not k.startswith("_")}
        safe_ns["abs"] = abs
        result = eval(expression, {"__builtins__": {}}, safe_ns)
        output = f"Result of '{expression}' = {result}"
        print(f"[TOOL: calculator] {output}")
        return output
    except Exception as e:
        error = f"Could not evaluate '{expression}': {e}"
        print(f"[TOOL: calculator] ERROR: {error}")
        return error


@tool
def get_definition(term: str) -> str:
    """Return a short definition of a technical term."""
    print(f"[TOOL: get_definition] term='{term}'")
    definitions = {
        "rag":          "RAG (Retrieval-Augmented Generation) combines document retrieval with LLM generation to answer questions using external knowledge.",
        "langchain":    "LangChain is a Python/TypeScript framework for building LLM-powered applications with chains, agents, and memory.",
        "langgraph":    "LangGraph is a LangChain extension for building stateful multi-actor workflows using a directed graph structure.",
        "embedding":    "An embedding is a high-dimensional numerical vector representing the semantic meaning of a piece of text.",
        "faiss":        "FAISS (Facebook AI Similarity Search) is an open-source library for efficient similarity search over dense vectors.",
        "agent":        "An agent is an LLM-powered system that autonomously chooses and calls tools to complete a task.",
        "vector store": "A vector store is a database optimized for storing and retrieving high-dimensional embedding vectors via similarity search.",
    }
    result = definitions.get(
        term.lower().strip(),
        f"No definition found for '{term}'. Known terms: rag, langchain, langgraph, embedding, faiss, agent, vector store"
    )
    print(f"[TOOL: get_definition] {result[:80]}...")
    return result


print("[TOOLS] Registered: get_weather | calculator | get_definition")

In [ ]:
# ── Guardrail ─────────────────────────────────────────────────────────────────
BLOCKED_KEYWORDS = ["bomb", "weapon", "hack", "exploit", "malware", "virus", "illegal", "drugs", "nsfw"]

def guardrail_check(user_input: str) -> tuple[bool, str]:
    """Returns (is_safe, reason). Blocks short inputs and harmful keywords."""
    print(f"[GUARDRAIL] Checking: '{user_input[:80]}'")
    if len(user_input.strip()) < 3:
        print("[GUARDRAIL] BLOCKED — too short")
        return False, "Input is too short. Please provide a meaningful question."
    for kw in BLOCKED_KEYWORDS:
        if kw in user_input.lower():
            print(f"[GUARDRAIL] BLOCKED — keyword='{kw}'")
            return False, f"Disallowed content detected: '{kw}'."
    print("[GUARDRAIL] PASSED")
    return True, "OK"

In [ ]:
# ── LangGraph Agent ───────────────────────────────────────────────────────────
from typing import Annotated, TypedDict
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
import operator

tools = [get_weather, calculator, get_definition]

agent_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    openai_api_key=os.environ["OPENAI_API_KEY"]
).bind_tools(tools)

AGENT_SYSTEM = SystemMessage(content=(
    "You are a helpful assistant with access to weather, calculator, and definition tools. "
    "Use tools when needed to answer accurately."
))


class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]


def call_llm(state: AgentState) -> AgentState:
    print(f"\n[LLM] Invoking with {len(state['messages'])} message(s)...")
    response = agent_llm.invoke([AGENT_SYSTEM] + state["messages"])
    if response.tool_calls:
        print(f"[LLM] Tool calls: {[tc['name'] for tc in response.tool_calls]}")
    else:
        print(f"[LLM] Final answer: {response.content[:80]}...")
    return {"messages": [response]}


def should_continue(state: AgentState) -> str:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        print("[ROUTER] → tools")
        return "tools"
    print("[ROUTER] → END")
    return END


tool_node = ToolNode(tools)

workflow = StateGraph(AgentState)
workflow.add_node("llm", call_llm)
workflow.add_node("tools", tool_node)
workflow.set_entry_point("llm")
workflow.add_conditional_edges("llm", should_continue)
workflow.add_edge("tools", "llm")

agent_app = workflow.compile()
print("[AGENT] Graph compiled: llm ↔ tools loop")

In [ ]:
# ── Visualize Graph ───────────────────────────────────────────────────────────
from IPython.display import Image, display, Markdown

def show_langgraph(app, title: str):
    display(Markdown(f"### {title}"))
    graph = app.get_graph()
    try:
        display(Image(graph.draw_mermaid_png()))
    except Exception as e:
        print(f"PNG render failed ({e}). Mermaid source:\n")
        print(graph.draw_mermaid())

show_langgraph(agent_app, "LangGraph Agent (llm ↔ tools)")

In [ ]:
# ── Run Agent ─────────────────────────────────────────────────────────────────
def run_agent(user_input: str) -> str:
    print(f"\n{'='*60}\n[AGENT] {user_input}\n{'='*60}")
    is_safe, reason = guardrail_check(user_input)
    if not is_safe:
        print(f"[BLOCKED] {reason}")
        return reason
    final_state = agent_app.invoke({"messages": [HumanMessage(content=user_input)]})
    answer = final_state["messages"][-1].content
    print(f"\n[ANSWER] {answer}")
    return answer


run_agent("What is the weather like in Sydney right now?")

In [ ]:
run_agent("What is 45 / 8500, then multiply that by 3?")

In [ ]:
run_agent("What is an Aeroplane? Give me a short definition.")

In [ ]:
# Guardrail test
run_agent("How do I hack into a system?")